In [ ]:
import json

import torch

from data import load_dataset
from segmentation_unet_train import run_unet_training
from unet import LightningUNet

## 1) Load configuration from JSON

In [ ]:
with open('manual_nucleolus_unet_config.json') as fd:
    config = json.load(fd)

dataset_paths = config['dataset_paths']
dataset_options = config['dataset_options']
network_options = config['network_options']

# network_options

## Alterative / first time: Manually specify config

In [ ]:
import albumentations as A

dataset_paths = [
    {
        'base_path': '/scratch/hoerl/nucleolus_dna_labelling/',
        'image_subfolder': 'tifs',
        'label_subfolder': 'labels_sparse',
        'image_file_pattern': "*.tif",
        'label_file_pattern': "*.tif"
    },
]

tr = A.Compose([
    A.RandomCrop(256, 256),
    A.SquareSymmetry(),
    A.ToTensorV2()
])

dataset_options = {
    'val_fraction': 0,
    'planeselect_min_labeled_pixels': 50,
    'planeselect_center_planes_fraction': 1.0,
    'plane_sliding_window': 1,
    'normalization_strategy': 'per_image',
    'augmentation': tr.to_dict(),
    'n_classes': 3,
    'sparse_labeling': True
}

network_options = {
    'unet_intermediate_channels': [32, 64, 128],
    'class_weights': [0.5, 1, 2],
    'early_stop_patience': 50,
}

### Optional: save config to JSON

In [ ]:
# combine into single config to save
config = {
    'dataset_paths': dataset_paths,
    'dataset_options': dataset_options,
    'network_options': network_options
}

with open('manual_nucleolus_unet_config.json', 'w') as fd:
    json.dump(config, fd, indent=4)

## 2) Load data

In [ ]:
dataset_train, dataset_val = load_dataset(dataset_paths, dataset_options)

In [ ]:
from matplotlib import pyplot as plt
from random import randint

img, mask = dataset_train[randint(0, len(dataset_train)-1)]

fig, axs = plt.subplots(ncols=2)
axs[0].imshow(img.max(axis=0)[0]) # max-project along z-axis (NOTE: returns max val and index, we only want the max val)
axs[1].imshow(mask.squeeze())

len(dataset_train)

## 3) Run Training

In [ ]:
run_unet_training(dataset_train, dataset_val, dataset_options, network_options)

## Apply model to images

In [ ]:
from lightning.pytorch.utilities.model_summary import ModelSummary

net_inference =  LightningUNet.load_from_checkpoint('lightning_logs/version_34/checkpoints/epoch=19-step=5660.ckpt').eval()
# net_inference =  LightningUNet.load_from_checkpoint('/Users/david/Desktop/jurkat_nucleolin/unet_nucleolus_001/checkpoints/epoch=299-step=3300.ckpt').eval()

ModelSummary(net_inference, max_depth=3)

In [ ]:
from tifffile import imread
import torch
from data import SparseLabeledImageDataset, NormalizationStrategy

test_file = '/Users/david/Desktop/nucleolus_dna_labelling/tifs/26am06-02-2_0012_ch1.tif'

img = imread(test_file)
img = SparseLabeledImageDataset._normalize_intensities(img, NormalizationStrategy.PER_IMAGE)

# add two dummy dimensions (batch size, channels)
img = torch.from_numpy(img).float()[:, torch.newaxis, torch.newaxis]
img.shape

In [ ]:
import nd2
import torch

from data import SparseLabeledImageDataset, NormalizationStrategy, sliding_window_planewise_padded
from scipy.ndimage import gaussian_filter

test_file = '/Volumes/tmp/Data for David/J1_U1GFP_488_mScar-D3_Dox_561_SiR-DNA_640_-Dox_sequence_KO-Dppa3/1.nd2'
img = nd2.imread(test_file, dask=True, xarray=True)
img = img.isel(C=2).values

# img = gaussian_filter(img, 0.5)

print(img.shape)

img = SparseLabeledImageDataset._normalize_intensities(img, NormalizationStrategy.PER_IMAGE)

sliding_window = 1
if sliding_window > 1:
    img = sliding_window_planewise_padded(img, sliding_window)

    # deliberately set other planes to 0
    img[:, [0,1,3,4]] = 0

    img = torch.from_numpy(img).float()[:, torch.newaxis, :]
else:
    img = torch.from_numpy(img).float()[:, torch.newaxis, torch.newaxis]

img.shape

In [ ]:
from lightning import pytorch as L

trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)

    if pred.shape[1] == 1:
        probs = torch.sigmoid(pred[:,0])
        pred_labels = probs > 0.5
    else:
        probs = torch.softmax(pred, 1)
        pred_labels = pred.argmax(1)

# ALTERNATIVE with full loader (different batch size, etc.):
# img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis]
# predict_ds = torch.utils.data.TensorDataset(img)
# predict_loader = torch.utils.data.DataLoader(predict_ds, 1)

In [ ]:
import napari
from skimage.morphology import closing, opening
from calmutils.morphology.structuring_elements import hypersphere_centered

# labels_postproc = opening(pred_labels.numpy(), hypersphere_centered(3, 2))
labels_postproc = pred_labels.numpy()

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_image(img[:,0,sliding_window//2])

# view predictions of a single class
# viewer.add_labels((pred_labels==2).int())

# ALTERNATIVE: all classes
viewer.add_labels(labels_postproc)


# img2 = nd2.imread(test_file, dask=True, xarray=True)
# img2 = img2.isel(C=0, P=0).values
# viewer.add_image(img2)


# viewer.add_image(probs[:,0])

In [ ]:
# TEST: replace first conv (from multiple z-plane channels to only middle) to get rid of sliding window for single-plane data?
conv1 = net_inference.unet.encoder['encoder_block_0'].steps[0]
weight_middle = conv1.weight.data[:,2:3]
bias = conv1.bias.data

net_inference.unet.encoder['encoder_block_0'].steps[0] = torch.nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, stride=1, padding=1)
net_inference.unet.encoder['encoder_block_0'].steps[0].weight.data = weight_middle
net_inference.unet.encoder['encoder_block_0'].steps[0].bias.data = bias

# save checkpoint manually (-> saves updated weight tensors)
# Note: trainer needs to have been used (for prediction)
trainer.save_checkpoint('example.ckpt')

# manually load saved checkpoint dict and replace hparams (so model can be loaded)
ckpt = torch.load('example.ckpt')
ckpt['hyper_parameters']['input_channels'] = 1
torch.save(ckpt, 'example.ckpt')